# IKG Column Lineage Master Auto Refresh

Parses all SQL files from the IKG GitLab repository (`develop` branch) and extracts **column-level lineage**.

**Outputs:** Excel `ikg_column_lineage_master_auto_refresh_<TS>.xlsx` + Greenplum table (DROP/RECREATE).

**Schema resolution:** Locally uses `LOCAL_SCHEMA_OVERRIDES` in Cell 2. In Airflow, reads from Airflow Variables.

## 1. Imports

In [ ]:
import os, sys, getpass
import pandas as pd
from pathlib import Path
from datetime import datetime
from IPython.display import display

SCRIPT_DIR = Path('.')
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from ikg_column_lineage_master_auto_refresh import (
    extract_lineage_from_sql, _load_local_sql_files, _fetch_sql_files_from_gitlab,
    save_to_excel, save_to_greenplum, ensure_greenplum_schema,
    get_private_token, COLUMN_ORDER, OUTPUT_TABLE, JINJA_SCHEMA_MAP,
)
import ikg_column_lineage_master_auto_refresh as _mod

print('Imports OK. COLUMN_ORDER:', COLUMN_ORDER)

## 2. Configuration & Schema Map

In [ ]:
USE_LOCAL_SQL  = True
LOCAL_SQL_PATH = str(Path('.') / 'dags/ikg/scripts/sql')
GITLAB_TOKEN   = None   # prompted if None and USE_LOCAL_SQL=False
GREENPLUM_PASSWORD = None
SAVE_TO_GREENPLUM  = False

# ── Local schema overrides ─────────────────────────────────────────────────
# Map Jinja param names → real schema names used for information_schema lookups.
# In Airflow these are read automatically from Airflow Variables.
LOCAL_SCHEMA_OVERRIDES = {
    'IKG_SCHEMA':            'core_ikg',
    'EDW_INPUT_SCHEMA':      'core_wma_shared',
    'EDW_VIEW_INPUT_SCHEMA': 'core_wma_shared',
    'IKG_VENDOR_SCHEMA':     'core_wma_shared',
    'MODEL_SCHEMA':          'core_model',
    'NLG_SCHEMA':            'core_nlg',
    'IKG_WEALTHX_SCHEMA':    'sandbox_prj_smart_relationship',
}
_mod.JINJA_SCHEMA_MAP.update(LOCAL_SCHEMA_OVERRIDES)

print(f'Mode: {"Local" if USE_LOCAL_SQL else "GitLab"}')
print('Schema map:', _mod.JINJA_SCHEMA_MAP)

## 3. Greenplum Connection (optional — enables `information_schema` lookups for SELECT *)

In [ ]:
GP_HOST, GP_PORT, GP_DB, GP_USER = 'greenplum-rdsp.zur.swissbank.com', 5432, 'gprdsp', 'ds_rdsp_dev'

try:
    _gp_pw = getpass.getpass(f'Greenplum password for {GP_USER} (Enter to skip): ')
    if _gp_pw:
        from sqlalchemy import create_engine
        _mod._DB_ENGINE = create_engine(f'postgresql://{GP_USER}:{_gp_pw}@{GP_HOST}:{GP_PORT}/{GP_DB}')
        if SAVE_TO_GREENPLUM:
            GREENPLUM_PASSWORD = _gp_pw
        print('Greenplum connected — info_schema lookups enabled.')
    else:
        print('Skipped — SELECT * will emit placeholder rows.')
except Exception as e:
    print(f'Connection failed: {e}')

## 4. Fetch SQL Files

In [ ]:
if USE_LOCAL_SQL:
    file_dict = _load_local_sql_files(LOCAL_SQL_PATH)
else:
    if GITLAB_TOKEN is None:
        GITLAB_TOKEN = getpass.getpass('GitLab private token: ')
    file_dict = _fetch_sql_files_from_gitlab(GITLAB_TOKEN)
print(f'Loaded: {len(file_dict)} SQL files')

## 5. Parse Lineage

In [ ]:
all_rows, parse_errors = [], []
for i, (fpath, content) in enumerate(file_dict.items(), 1):
    if i % 100 == 0: print(f'  {i}/{len(file_dict)}...', end='\r')
    try:
        all_rows.extend(extract_lineage_from_sql(content, Path(fpath).name, fpath, Path(fpath).parent.name))
    except Exception as e:
        parse_errors.append({'file': fpath, 'error': str(e)})
print(f'Done. Records: {len(all_rows)}, Errors: {len(parse_errors)}')
if parse_errors: display(pd.DataFrame(parse_errors))

## 6. Build DataFrame

In [ ]:
df = pd.DataFrame(all_rows, columns=COLUMN_ORDER).drop_duplicates()
df['current_date_time'] = pd.to_datetime(df['current_date_time'])
print(f'Deduped: {len(df):,} rows | Target tables: {df["target_table"].nunique()} | Source tables: {df["source_table"].nunique()}')
display(df['sql_process'].value_counts().reset_index())
df.head(5)

## 7. Analysis

In [ ]:
LOOKUP_TABLE = 'fee_waiver_rma_ikg'
result = df[df['target_table'] == LOOKUP_TABLE]
print(f'Lineage for [{LOOKUP_TABLE}]: {len(result)} rows')
display(result[['target_column','source_table','source_schema','source_column','logic','sql_process']].head(30))

## 8. Export Excel

In [ ]:
out = f'ikg_column_lineage_master_auto_refresh_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
save_to_excel(df, out)
print(f'Saved: {out}')

## 9. Save to Greenplum

In [ ]:
if SAVE_TO_GREENPLUM:
    schema = ensure_greenplum_schema()
    if not GREENPLUM_PASSWORD:
        GREENPLUM_PASSWORD = getpass.getpass('Greenplum password: ')
    ok = save_to_greenplum(df, schema, GREENPLUM_PASSWORD)
    print(f'{"Saved" if ok else "FAILED"} → {schema}.{OUTPUT_TABLE}')
else:
    print('Skipped (SAVE_TO_GREENPLUM=False)')

## 10. Schema Reference

| Column | Description |
|---|---|
| `file` | SQL filename with extension |
| `path` | Relative repo path |
| `target_table` | Final output table (= filename stem) |
| `target_schema` | Schema of target_table |
| `process` | DAG group folder |
| `current_date_time` | Extraction timestamp |
| `sub_target_table` | Every table/CTE created in this script |
| `sub_target_schema` | Schema of sub_target_table |
| `target_column` | Column written into sub_target_table |
| `source_table` | Real source table |
| `source_schema` | Schema of source_table |
| `source_column` | Source column |
| `logic` | Raw SQL expression |
| `sql_process` | `select` `select-value` `select*` `join` `where` `having` `where-subquery` |